In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np

In [40]:
#Loading and making the melamchi dataset for flood prediction 

df_melamchi_waterFlow = pd.read_csv("Datasets/melamchi_waterflow.csv")
df_melamchi_waterFlow.rename(columns={'Values': 'WaterFlow','Dates':'Date'},inplace=True)

df_melamchi_weather = pd.read_csv('Datasets/melamchi_weather.csv')
df_melamchi_weather.rename(columns={'YEAR':'Date'},inplace=True)

date_string_weather = df_melamchi_weather['Date'].astype(str) + ' '+ df_melamchi_weather['DOY'].astype(str) 

df_melamchi_waterFlow['Date'] = pd.to_datetime(df_melamchi_waterFlow['Date'])
df_melamchi_weather['Date'] = pd.to_datetime(date_string_weather, format="%Y %j")

df_melamchi_weather.drop(columns='DOY',inplace=True)

df_melamchi = pd.merge_asof(df_melamchi_weather,df_melamchi_waterFlow,on='Date')
df_melamchi.drop(columns='Date',inplace=True)

df_melamchi.rename(columns={
    'PRECTOTCORR': 'Rainfall', 
    'T2M': 'Temperature',
    'RH2M': 'Relative_Humidity'
},inplace=True)

#Creating flood occurrence label
flood_threshold = df_melamchi['WaterFlow'].quantile(0.95)
df_melamchi['floodOccurrence'] = (df_melamchi['WaterFlow'] > flood_threshold).astype(int)

#Separating the data for training and validation
y = df_melamchi.pop('floodOccurrence')
X = df_melamchi

separating_idx = int(len(df_melamchi) * 0.8)
X_train_raw  = X.iloc[:separating_idx]
y_train_raw = y.iloc[:separating_idx]

X_val_raw = X.iloc[separating_idx:]
y_val_raw = y.iloc[separating_idx:]


#Scaling the data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

#Changing to tensors
X_train_tensors = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensors = torch.tensor(y_train_raw.values,dtype=torch.float32).unsqueeze(1) #to make the size [N,1]

X_val_tensors = torch.tensor(X_val_scaled,dtype=torch.float32)
y_val_tensors = torch.tensor(y_val_raw.values,dtype=torch.float32).unsqueeze(1) 


In [43]:
#model definition
class NeuralModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.hidden1 = nn.Linear(input_size,64)
        self.hidden2 = nn.Linear(64,64)
        self.output = nn.Linear(64,1)
        self.relu = nn.ReLU()
        
    def forward(self,x):
        x = self.hidden1(x)
        x = self.relu(x)
        x = self.hidden2(x)
        x = self.relu(x)
        return self.output(x)
 
#training process
def train_model(model,X_train,X_val,y_train,y_val,epochs,lr):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(),lr=lr)
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        predicted = model(X_train)
        train_loss = criterion(predicted,y_train)
        
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_predicted = model(X_val)
            val_loss = criterion(val_predicted,y_val)
            
        # Print metrics every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{epochs}] -> Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
    

In [45]:
#Flood prediction model
input_features = X_train_tensors.shape[1]
flood_model = NeuralModel(input_size=input_features)
train_model(
    model=flood_model,
    X_train=X_train_tensors,
    X_val=X_val_tensors,
    y_train=y_train_tensors,
    y_val=y_val_tensors,
    epochs=200,
    lr=0.001
)

Epoch [1/200] -> Train Loss: 0.6498 | Val Loss: 0.6388
Epoch [5/200] -> Train Loss: 0.5832 | Val Loss: 0.5712
Epoch [10/200] -> Train Loss: 0.5042 | Val Loss: 0.4917
Epoch [15/200] -> Train Loss: 0.4302 | Val Loss: 0.4174
Epoch [20/200] -> Train Loss: 0.3619 | Val Loss: 0.3497
Epoch [25/200] -> Train Loss: 0.3005 | Val Loss: 0.2896
Epoch [30/200] -> Train Loss: 0.2483 | Val Loss: 0.2392
Epoch [35/200] -> Train Loss: 0.2064 | Val Loss: 0.1989
Epoch [40/200] -> Train Loss: 0.1744 | Val Loss: 0.1680
Epoch [45/200] -> Train Loss: 0.1502 | Val Loss: 0.1450
Epoch [50/200] -> Train Loss: 0.1318 | Val Loss: 0.1279
Epoch [55/200] -> Train Loss: 0.1173 | Val Loss: 0.1151
Epoch [60/200] -> Train Loss: 0.1056 | Val Loss: 0.1053
Epoch [65/200] -> Train Loss: 0.0957 | Val Loss: 0.0970
Epoch [70/200] -> Train Loss: 0.0871 | Val Loss: 0.0896
Epoch [75/200] -> Train Loss: 0.0795 | Val Loss: 0.0828
Epoch [80/200] -> Train Loss: 0.0727 | Val Loss: 0.0765
Epoch [85/200] -> Train Loss: 0.0667 | Val Loss: 0

In [47]:
torch.save(flood_model.state_dict(),"flood_model.pth")